## EDA on the HIPE-OCRepair-2026 dataset

In [1]:
!chmod +x ./scripts/*
!./scripts/download_dataset.sh

[INFO] Directory 'HIPE-OCRepair-2026-data' already exists.
Already up to date.
[SUCCESS] Repository successfully updated.


In [5]:
import json
import glob
import pandas as pd
import re

In [22]:
#configs
data_dir = "./HIPE-OCRepair-2026-data/data/v0.9"
split_name = "test" # change to train, dev, test

search_pattern = f"{data_dir}/**/*_{split_name}*.jsonl"
file_paths = glob.glob(search_pattern, recursive=True)

file_paths

['./HIPE-OCRepair-2026-data/data/v0.9/impresso-nzz/de/hipe-ocrepair-bench_v0.9_impresso-nzz_v1.1_test_de.jsonl',
 './HIPE-OCRepair-2026-data/data/v0.9/impresso-snippets/en/hipe-ocrepair-bench_v0.9_impresso-snippets_v1.0_test_en.jsonl',
 './HIPE-OCRepair-2026-data/data/v0.9/impresso-snippets/de/hipe-ocrepair-bench_v0.9_impresso-snippets_v1.0_test_de.jsonl',
 './HIPE-OCRepair-2026-data/data/v0.9/impresso-snippets/fr/hipe-ocrepair-bench_v0.9_impresso-snippets_v1.0_test_fr.jsonl',
 './HIPE-OCRepair-2026-data/data/v0.9/overproof/en/hipe-ocrepair-bench_v0.9_overproof-combined_v1.0_test_en.jsonl',
 './HIPE-OCRepair-2026-data/data/v0.9/dta19/de/hipe-ocrepair-bench_v0.9_dta19-l2_v0.1_test-unmatched_de.jsonl',
 './HIPE-OCRepair-2026-data/data/v0.9/dta19/de/hipe-ocrepair-bench_v0.9_dta19-l0_v0.1_test-unmatched_de.jsonl',
 './HIPE-OCRepair-2026-data/data/v0.9/dta19/de/hipe-ocrepair-bench_v0.9_dta19-l1_v0.1_test-unmatched_de.jsonl',
 './HIPE-OCRepair-2026-data/data/v0.9/icdar2017/en/hipe-ocrepair-b

In [23]:
extracted_data = []

for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            doc = json.loads(line)
            
            metadata = doc.get("document_metadata", {})
            ocr_hyp = doc.get("ocr_hypothesis", {})
            ocr_quality = ocr_hyp.get("quality_report", {})
            gt = doc.get("ground_truth", {})
            
            extracted_data.append({
                "dataset": metadata.get("primary_dataset_name", "unknown"),
                "date": metadata.get("date", ""),
                "language": metadata.get("language", "unknown"),
                "doc_type": metadata.get("document_type", ""),
                "publication": metadata.get("publication_title", ""),
                "scope": metadata.get("transcription_unit_scope", ""),
                
                "cer": float(ocr_quality.get("cer")) if ocr_quality.get("cer") is not None else None,
                "wer": float(ocr_quality.get("wer")) if ocr_quality.get("wer") is not None else None,
                "char_errors": ocr_quality.get("nb_char_errors"),
                "char_subst": ocr_quality.get("nb_char_substitutions"),
                "char_del": ocr_quality.get("nb_char_deletions"),
                "char_ins": ocr_quality.get("nb_char_insertions"),
                "ocr_score": float(ocr_quality.get("ocr_quality_score")) if ocr_quality.get("ocr_quality_score") is not None else None,
                
                "ocr_chars": ocr_hyp.get("num_chars", 0),
                "gt_chars": gt.get("num_chars", 1),
                
                "num_paragraphs": len(gt.get("paragraph_offsets", [])),
                "num_lines": len(gt.get("line_offsets", [])),
                "num_sentences": len(gt.get("sentence_offsets", [])),
            })
            
df = pd.DataFrame(extracted_data)
df.loc[(df['dataset'] == 'dta19') & (df['wer'] == 0), 'wer'] = None

def extract_year(date_str):
    if pd.isna(date_str) or date_str == "n/a":
        return None
    match = re.search(r'(\d{4})', str(date_str))
    return int(match.group(1)) if match else None
   
df['year'] = df['date'].apply(extract_year)


def format_stat(series):
    series = pd.to_numeric(series, errors='coerce').dropna()
    if series.empty: return "N/A"
    return f"{series.mean():.4f} ± {series.std():.4f}"

def format_langs(series):
    counts = series.value_counts()
    return ", ".join([f"{lang}({count})" for lang, count in counts.items()])

summary = df.groupby("dataset").agg({
    "year": lambda x: f"{int(x.min())} - {int(x.max())}" if not x.dropna().empty else "N/A",
    "language": format_langs,
    "cer": format_stat,
    "wer": format_stat,
    "ocr_score": format_stat
}).reset_index()

summary.columns = ["Dataset", "Period", "Languages (Samples)", "CER (mean ± std)", "WER (mean ± std)", "OCR Score (mean ± std)"]
summary

,Dataset,Period,Languages (Samples),CER (mean ± std),WER (mean ± std),OCR Score (mean ± std)
0,dta19,1802 - 1863,de(240),0.0330 ± 0.0272,N/A,N/A
1,icdar2017,N/A,"en(101), fr(100)",0.0312 ± 0.0253,0.1375 ± 0.0860,0.9106 ± 0.0525
2,impresso-nzz,1780 - 1939,de(17),0.0989 ± 0.1828,0.3836 ± 0.6777,0.7771 ± 0.0684
3,impresso-snippets,1801 - 1959,"en(100), de(100), fr(100)",0.0365 ± 0.0301,0.2257 ± 0.2258,N/A
4,overproof-combined,1859 - 1949,en(32),0.0903 ± 0.0708,0.3145 ± 0.1821,0.8838 ± 0.0775


You can then rerun again but change the split_name to "dev" to EDA the validation set